In [19]:
#!pip install pymongo

In [20]:
import requests
from pymongo import MongoClient, ASCENDING


In [21]:
# MongoDB Config

MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "ny_project"
COLLECTION = "acs_zip_population"

# Years to Fetch
YEARS = [2021, 2022, 2023, 2024, 2025]

In [ ]:
# Helper Functions
def is_ny_zip(zip_code: str) -> bool:
    """
    Check if ZIP code belongs to New York state.
    NY ZIP codes range from 10000 to 14999.
    """
    if not zip_code:
        return False
    return zip_code.startswith(("10", "11", "12", "13", "14"))


In [23]:
# Fetch Data from ACS API

def fetch_year_data(year: int):
    """
    Fetch ACS ZIP population data for a given year.
    Only NY ZIP codes are kept.
    """
    api_url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": "NAME,B01003_001E",
        "for": "zip code tabulation area:*"
    }

    print(f"\nFetching year {year}...")

    try:
        response = requests.get(api_url, params=params, timeout=60)
        if response.status_code != 200:
            print(f"Year {year} not available (status {response.status_code})")
            return []

        data = response.json()
        if not data or len(data) < 2:
            print(f"Year {year} returned empty or invalid data")
            return []

        headers = data[0]
        rows = data[1:]
        docs = []

        for row in rows:
            rec = dict(zip(headers, row))
            zip_code = rec.get("zip code tabulation area")
            if not zip_code or not is_ny_zip(zip_code):
                continue

            try:
                population = int(rec.get("B01003_001E"))
            except (TypeError, ValueError):
                population = None

            docs.append({
                "year": year,
                "state_2": "NY",
                "zip_code": zip_code,
                "population": population,
                "data_class": "Population",
                "data_field": "B01003_001E",
                "data_field_display_name": "Total Population",
                "data_stream": "ACS_API_JSON"
            })

        print(f"NY ZIP rows for {year}: {len(docs)}")
        return docs

    except requests.exceptions.RequestException as e:
        print(f"Error fetching year {year}: {type(e).__name__}: {e}")
        return []
    except Exception as e:
        print(f"Unexpected error for year {year}: {type(e).__name__}: {e}")
        return []


In [24]:
# MongoDB Storage Function

def store_docs(docs, mongo_uri, db_name, collection_name):
    """
    Upsert a list of documents into MongoDB.
    """
    client = None
    total_upserts = 0

    try:
        client = MongoClient(mongo_uri)
        collection = client[db_name][collection_name]

        # Prevent duplicates across years
        try:
            collection.create_index(
                [("year", ASCENDING), ("zip_code", ASCENDING)],
                unique=True
            )
        except Exception as e:
            print(f"Index creation warning: {type(e).__name__}: {e}")

        for doc in docs:
            try:
                collection.update_one(
                    {"year": doc["year"], "zip_code": doc["zip_code"]},
                    {"$set": doc},
                    upsert=True
                )
                total_upserts += 1
            except Exception as e:
                print(f"Error upserting document: {type(e).__name__}: {e}")
                continue

    except Exception as e:
        print(f"MongoDB error: {type(e).__name__}: {e}")
    finally:
        if client:
            client.close()
    
    return total_upserts


In [25]:
# Main Execution
def main():
    all_docs = []

    for year in YEARS:
        docs = fetch_year_data(year)
        all_docs.extend(docs)

    total_upserts = store_docs(all_docs, MONGO_URI, DB_NAME, COLLECTION)
    print(f"\nTotal documents upserted: {total_upserts}")
    print("Done")

# Run the main function
if __name__ == "__main__":
    main()



Fetching year 2021...
NY ZIP rows for 2021: 1825

Fetching year 2022...
NY ZIP rows for 2022: 1825

Fetching year 2023...
NY ZIP rows for 2023: 1823

Fetching year 2024...
Year 2024 not available (status 404)

Fetching year 2025...
Year 2025 not available (status 404)

Total documents upserted: 5473
Done
